In [126]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import accuracy_score, mean_squared_error

In [128]:
train_df = pd.read_excel("Sample_arvyax_reflective_dataset.xlsx")

test_df = pd.read_excel("arvyax_test_inputs_120.xlsx")

In [129]:
train_df.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3
2,3,The forest session slowed my thoughts and I fe...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5


In [132]:
train_df.columns

Index(['id', 'journal_text', 'ambience_type', 'duration_min', 'sleep_hours',
       'energy_level', 'stress_level', 'time_of_day', 'previous_day_mood',
       'face_emotion_hint', 'reflection_quality', 'emotional_state',
       'intensity'],
      dtype='object')

In [134]:
train_df.isnull().sum()

id                      0
journal_text            0
ambience_type           0
duration_min            0
sleep_hours             7
energy_level            0
stress_level            0
time_of_day             0
previous_day_mood      15
face_emotion_hint     123
reflection_quality      0
emotional_state         0
intensity               0
dtype: int64

In [136]:
train_df['sleep_hours'] = train_df['sleep_hours'].fillna(train_df['sleep_hours'].mean())

train_df['previous_day_mood'] = train_df['previous_day_mood'].fillna('unknown')

train_df['face_emotion_hint'] = train_df['face_emotion_hint'].fillna('unknown')

In [138]:
train_df.isnull().sum()

id                    0
journal_text          0
ambience_type         0
duration_min          0
sleep_hours           0
energy_level          0
stress_level          0
time_of_day           0
previous_day_mood     0
face_emotion_hint     0
reflection_quality    0
emotional_state       0
intensity             0
dtype: int64

In [140]:
train_df['journal_text'] = train_df['journal_text'].str.lower()
train_df['journal_text'] = train_df['journal_text'].str.replace('[^a-zA-Z ]', '', regex=True)

In [142]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000)

X_text = tfidf.fit_transform(train_df['journal_text'])

In [144]:
meta_cols = [
    'sleep_hours',
    'energy_level',
    'stress_level',
    'duration_min'
]

X_meta = train_df[meta_cols]

In [146]:
from scipy.sparse import hstack

X = hstack([X_text, X_meta])

In [148]:
y_state = train_df['emotional_state']
y_intensity = train_df['intensity']

In [150]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_state_train, y_state_test = train_test_split(
    X, y_state, test_size=0.2, random_state=42
)

In [152]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)

clf.fit(X_train, y_state_train)

LogisticRegression(max_iter=1000)

In [153]:
y_pred = clf.predict(X_test)

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_state_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.625


In [154]:
from sklearn.ensemble import RandomForestRegressor

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X, y_intensity, test_size=0.2, random_state=42
)

reg = RandomForestRegressor()

reg.fit(X_train_i, y_train_i)

RandomForestRegressor()

In [157]:
y_pred_i = reg.predict(X_test_i)

from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test_i, y_pred_i)
print("MSE:", mse)

MSE: 2.276340053703704


In [158]:
y_pred_i = y_pred_i.round().clip(1, 5)

In [159]:
def decide_action(state, intensity, energy, stress, time_of_day):
    
    if stress >= 4:
        return "box_breathing", "now"
    
    if energy <= 2:
        return "rest", "later_today"
    
    if state in ["calm", "focused"] and energy >= 3:
        return "deep_work", "now"
    
    if state in ["sad", "overwhelmed"]:
        return "journaling", "within_15_min"
    
    if time_of_day == "night":
        return "light_planning", "tomorrow_morning"
    
    return "pause", "later_today"

In [160]:
print(decide_action("anxious", 4, 2, 5, "evening"))

('box_breathing', 'now')


In [161]:
probs = clf.predict_proba(X_test)
confidence_scores = probs.max(axis=1)

uncertain_flag = (confidence_scores < 0.6).astype(int)

In [162]:
test_df['sleep_hours'] = test_df['sleep_hours'].fillna(train_df['sleep_hours'].mean())
test_df['previous_day_mood'] = test_df['previous_day_mood'].fillna('unknown')
test_df['face_emotion_hint'] = test_df['face_emotion_hint'].fillna('unknown')

In [163]:
test_df['journal_text'] = test_df['journal_text'].str.lower()
test_df['journal_text'] = test_df['journal_text'].str.replace('[^a-zA-Z ]', '', regex=True)

In [164]:
X_text_test = tfidf.transform(test_df['journal_text'])

X_meta_test = test_df[meta_cols]

from scipy.sparse import hstack
X_test_final = hstack([X_text_test, X_meta_test])

In [165]:
pred_state = clf.predict(X_test_final)

pred_intensity = reg.predict(X_test_final)
pred_intensity = pred_intensity.round().clip(1, 5)

In [166]:
probs_test = clf.predict_proba(X_test_final)
confidence = probs_test.max(axis=1)

uncertain_flag = (confidence < 0.6).astype(int)

In [167]:
actions = []
timings = []

for i in range(len(test_df)):
    state = pred_state[i]
    intensity = int(pred_intensity[i])
    energy = test_df.iloc[i]['energy_level']
    stress = test_df.iloc[i]['stress_level']
    time_of_day = test_df.iloc[i]['time_of_day']
    
    action, timing = decide_action(state, intensity, energy, stress, time_of_day)
    
    actions.append(action)
    timings.append(timing)

In [168]:
output_df = pd.DataFrame({
    'id': test_df['id'],
    'predicted_state': pred_state,
    'predicted_intensity': pred_intensity,
    'confidence': confidence,
    'uncertain_flag': uncertain_flag,
    'what_to_do': actions,
    'when_to_do': timings
})

In [169]:
output_df.to_csv("predictions.csv", index=False)

In [170]:
output_df.head()

,id,predicted_state,predicted_intensity,confidence,uncertain_flag,what_to_do,when_to_do
0,10001,calm,3.0,0.288036,1,deep_work,now
1,10002,mixed,4.0,0.268719,1,rest,later_today
2,10003,focused,4.0,0.287567,1,box_breathing,now
3,10004,focused,3.0,0.506189,1,rest,later_today
4,10005,neutral,3.0,0.269875,1,box_breathing,now


## Error Analysis


In [172]:
y_pred = clf.predict(X_test)

error_df = pd.DataFrame({
    'text': train_df.iloc[y_state_test.index]['journal_text'],
    'actual': y_state_test,
    'predicted': y_pred
})

errors = error_df[error_df['actual'] != error_df['predicted']]

errors.head(10)

,text,actual,predicted
1178,lowkey felt pretty even but this was better th...,neutral,mixed
1120,lowkey felt locked in for a bit but mountain v...,focused,restless
855,honestly i felt lighter than before i couldnt ...,calm,restless
820,that helped a little,calm,restless
523,felt heavy,calm,overwhelmed
765,got distracted again,mixed,focused
838,i guess mind was all over the place,focused,restless
380,gradually my breathing slowed down even if onl...,calm,restless
609,honestly still anxious a bit after some time k...,focused,restless
676,okay session,neutral,calm


# Error Analysis

## Overview

The model was evaluated on validation data. The errors reveal key limitations of TF-IDF-based models when dealing with ambiguous, short, and context-dependent emotional expressions.

---

## Case 1: Ambiguous Emotional Expression

Text: “lowkey felt pretty even but this was better…”
Actual: neutral
Predicted: mixed

Reason: The phrase “lowkey” and “better” introduces slight positivity, causing confusion between neutral and mixed.
Improvement: Use contextual embeddings to capture subtle tone differences.

---

## Case 2: Contextual Misinterpretation

Text: “lowkey felt locked in for a bit but mountain v…”
Actual: focused
Predicted: restless

Reason: The model likely focused on fragmented or unclear words, missing the “locked in” signal of focus.
Improvement: Better text cleaning and phrase-level understanding.

---

## Case 3: Positive Shift Misread

Text: “honestly i felt lighter than before i couldn’t …”
Actual: calm
Predicted: restless

Reason: “lighter” indicates improvement, but earlier negative tone may have influenced prediction.
Improvement: Use sequence-aware models (BERT).

---

## Case 4: Very Short Text

Text: “that helped a little”
Actual: calm
Predicted: restless

Reason: Insufficient information; model cannot infer emotional state reliably.
Improvement: Use metadata fallback + uncertainty flag.

---

## Case 5: Weak Emotional Signal

Text: “felt heavy”
Actual: calm
Predicted: overwhelmed

Reason: “heavy” is often associated with negative emotions, causing overestimation.
Improvement: Combine with stress/energy features for balance.

---

## Case 6: Activity vs Emotion Confusion

Text: “got distracted again”
Actual: mixed
Predicted: focused

Reason: Model misinterprets behavioral signal (distraction) as cognitive engagement.
Improvement: Incorporate behavioral features explicitly.

---

## Case 7: Conflicting Signals

Text: “i guess mind was all over the place”
Actual: focused
Predicted: restless

Reason: Text suggests restlessness, but label is focused — possible label noise.
Improvement: Handle noisy labels with robust training.

---

## Case 8: Subtle Calm State

Text: “gradually my breathing slowed down even if onl…”
Actual: calm
Predicted: restless

Reason: Model fails to capture calming progression due to lack of temporal understanding.
Improvement: Use sequence models.

---

## Case 9: Residual Anxiety

Text: “honestly still anxious a bit after some time…”
Actual: focused
Predicted: restless

Reason: Strong keyword “anxious” dominates prediction despite functional state.
Improvement: Reduce keyword bias via contextual models.

---

## Case 10: Extremely Short Input

Text: “okay session”
Actual: neutral
Predicted: calm

Reason: Very little signal; model defaults to common class.
Improvement: Use uncertainty flag for short inputs.

---

## Key Insights

* **Ambiguity is the biggest challenge**: Short and vague reflections reduce model reliability
* **Keyword bias**: Model over-relies on words like “anxious”, “heavy”
* **Lack of context understanding**: TF-IDF cannot capture sentence meaning
* **Metadata is underutilized**: Could help resolve ambiguity
* **Label noise exists**: Some labels contradict text
* **Short inputs need fallback strategies**

---

## Conclusion

The model performs reasonably well on clear inputs but struggles with:

* vague reflections
* conflicting signals
* short text

Future improvements include:

* transformer-based models (BERT)
* better metadata fusion
* uncertainty-aware decision making
